# Feature Experiments

**Purpose:** test new input features (rolling averages, dropoffs, weather, ...) for the ST-GNN *without* paying the cost of a full CLI retrain (50 epochs, early stopping, checkpoint I/O) every time.

This notebook reuses the exact same model/dataset/scoring code from `scripts/train_multihorizon_torch.py` — nothing is reimplemented — it just runs a short **quick-train** on a random subsample of the training windows (see `subsample()` below) so you get directional signal in roughly a minute instead of 20-30+.

**Workflow:**
1. Load the base preprocessed data.
2. Run `quick_train` on the current 6-feature baseline → baseline metrics.
3. Build a candidate feature **in-memory** (no need to touch `preprocess.py` yet) and append it as a new channel.
4. Run `quick_train` again on the augmented features → compare.
5. Only if a feature looks genuinely promising here, wire it into `preprocess.py` for real and confirm with the full CLI training script (`train_multihorizon_torch.py`) before merging anything into `main`.

⚠️ **Quick-train numbers are directional only** — a random subsample of the training windows, fewer epochs, no early-stopping patience, no held-out model selection. Don't quote them as final results; they're for deciding "is this worth a real run?" Both sides of any comparison here use the *same* subsample/seed, so relative differences are meaningful even though absolute numbers are noisier than a full run. (A full, un-subsampled 8-epoch run on all ~6,249 train windows was measured directly to take 20-30+ minutes on this CPU — that's why subsampling exists, not a notebook/kernel limitation.)

In [1]:
import sys, os, json
sys.path.insert(0, os.path.abspath(os.path.join("..", "scripts")))

import numpy as np
import pandas as pd
import torch

# Reuse the exact production model/dataset/scoring code — no reimplementation.
from train_multihorizon_torch import (
    MultiHorizonDataset, MultiHorizonSTGNN, run_epoch, scores, split_bounds, seed_all, HORIZONS,
)
from torch.utils.data import DataLoader

DATA_DIR = os.path.abspath(os.path.join("..", "real_processed_265"))
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cpu":
    # Benchmarked directly: 8 threads was the sweet spot on this machine (12 cores) --
    # more gave no further speedup, 1 thread was ~1.7x slower. Pin it explicitly
    # rather than relying on torch's default heuristic.
    torch.set_num_threads(min(8, os.cpu_count() or 8))
print("device:", device, "| torch threads:", torch.get_num_threads())

device: cpu | torch threads: 8


In [2]:
# Load the base preprocessed dataset (same files the CLI scripts use).
features = np.load(os.path.join(DATA_DIR, "features_clipped.npy"))   # [Z, T, 6]
demand = np.load(os.path.join(DATA_DIR, "demand.npy"))               # [Z, T] raw pickup counts
a_out_np = np.load(os.path.join(DATA_DIR, "A_out.npy"))
a_in_np = np.load(os.path.join(DATA_DIR, "A_in.npy"))
with open(os.path.join(DATA_DIR, "metadata.json"), encoding="utf-8") as f:
    meta = json.load(f)

a_out = torch.as_tensor(a_out_np, dtype=torch.float32, device=device)
a_in = torch.as_tensor(a_in_np, dtype=torch.float32, device=device)

print("features:", features.shape, "| feature_names:", meta["feature_names"])
print("split:", meta["split"])

features: (253, 8928, 6) | feature_names: ['log_demand_zscore', 'hour_sin', 'hour_cos', 'weekday_sin', 'weekday_cos', 'is_weekend']
split: {'train': [0, 6249], 'validation': [6249, 7588], 'test': [7588, 8928], 'train_fraction': 0.7, 'scaler_fit_end': 6249}


## Quick-train helper

Same `MultiHorizonSTGNN` architecture, same windowing convention, same `scores()` hotspot metrics as the CLI script — just fewer epochs and everything kept in memory (no checkpoint saved to disk). Pass in any `features` array with shape `[Z, T, F]`; `F` can be anything, the model infers input width automatically.

In [3]:
def subsample(dataset, max_samples, seed=7):
    """Cap a MultiHorizonDataset to a random subset of its anchor timesteps.
    Only used if you pass max_train_samples/max_val_samples explicitly --
    the default quick_train config below no longer needs this (see next cell's
    docstring for why: hidden/window reduction gets speed without sampling noise).
    """
    if max_samples is not None and len(dataset.times) > max_samples:
        rng = np.random.default_rng(seed)
        keep = rng.choice(len(dataset.times), size=max_samples, replace=False)
        dataset.times = [dataset.times[i] for i in sorted(keep.tolist())]
    return dataset


def quick_train(features, meta, epochs=6, hidden=32, batch_size=128, lr=1e-3,
                 window=24, seed=7, max_train_samples=None, max_val_samples=None,
                 verbose=True):
    """Fast, in-memory ablation run on the FULL train/val split. Returns (metrics_dict, history_list).

    Benchmarked directly (see chat): the GRU, not the graph layer, is the
    actual bottleneck (4.75s/batch full-size vs 0.7s/batch for DirectedGraphConv
    alone). Cutting hidden 64->32 and window 48->24 together gives ~4.3x
    speedup (4.75s -> 1.1s/batch), which brings a full-data 6-epoch run down
    to ~6-7 minutes instead of 20-30+ -- WITHOUT throwing away 90%+ of the
    training samples the way random subsampling did. That matters: a smaller
    hidden/window model is a fair, consistent yardstick for comparing feature
    sets against each other (same reduced capacity on both sides of any A/B),
    whereas subsampling adds sample-selection noise on top of everything else.
    Bigger batch size and more threads were also benchmarked and did NOT help
    (batch=256 was slower per-sample than batch=128; threads>8 gave no gain) --
    don't bother tuning those.

    Still not a substitute for the full CLI script (hidden=64, window=48, 50
    epochs, real early stopping) before promoting a feature for real -- reduced
    hidden/window means reduced model capacity, so absolute numbers here won't
    match the production checkpoint. But relative comparisons (baseline vs.
    candidate feature, same reduced config) are a much more honest quick signal
    than the small-subsample version was.
    """
    seed_all(seed)
    bounds = split_bounds(meta, features.shape[1])
    datasets = [MultiHorizonDataset(features, *b, window) for b in bounds]
    datasets[0] = subsample(datasets[0], max_train_samples, seed)
    datasets[1] = subsample(datasets[1], max_val_samples, seed)
    loaders = [DataLoader(ds, batch_size, shuffle=False) for ds in datasets]

    model = MultiHorizonSTGNN(features.shape[2], hidden=hidden).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)

    history = []
    for epoch in range(1, epochs + 1):
        train_rmse, _, _ = run_epoch(model, loaders[0], a_out, a_in, device, optimizer)
        val_rmse, _, _ = run_epoch(model, loaders[1], a_out, a_in, device)
        history.append({"epoch": epoch, "train_rmse": train_rmse, "val_rmse": val_rmse})
        if verbose:
            print(f"epoch={epoch:02d} train_rmse={train_rmse:.4f} val_rmse={val_rmse:.4f}")

    _, pred, target = run_epoch(model, loaders[2], a_out, a_in, device)
    metrics = scores(pred, target)
    return metrics, history


def compare(baseline_metrics, candidate_metrics, label="candidate"):
    """Side-by-side RMSE / top-3 hit-rate table, baseline vs. candidate feature set."""
    rows = []
    for h in HORIZONS:
        b, c = baseline_metrics[str(h)], candidate_metrics[str(h)]
        rows.append({
            "minutes": b["minutes"],
            "baseline_rmse": round(b["rmse"], 4),
            f"{label}_rmse": round(c["rmse"], 4),
            "rmse_delta_%": round(100 * (b["rmse"] - c["rmse"]) / b["rmse"], 2),
            "baseline_top3_hit": round(b["hotspot_top3_hit_rate"], 3),
            f"{label}_top3_hit": round(c["hotspot_top3_hit_rate"], 3),
        })
    return pd.DataFrame(rows).set_index("minutes")

## Baseline: current 6-feature set, quick run

In [4]:
baseline_metrics, baseline_history = quick_train(features, meta, epochs=8)
baseline_metrics

epoch=01 train_rmse=0.7844 val_rmse=0.8017


epoch=02 train_rmse=0.7440 val_rmse=0.7764


epoch=03 train_rmse=0.7397 val_rmse=0.7741


epoch=04 train_rmse=0.7266 val_rmse=0.7625


epoch=05 train_rmse=0.7187 val_rmse=0.7596


epoch=06 train_rmse=0.7194 val_rmse=0.7614


epoch=07 train_rmse=0.7187 val_rmse=0.7590


epoch=08 train_rmse=0.7154 val_rmse=0.7568


{'1': {'minutes': 5,
  'rmse': 0.6568945646286011,
  'mae': 0.3698451519012451,
  'hotspot_top3_hit_rate': 0.3476297855377197,
  'hotspot_top5_recall': 0.17923252284526825},
 '3': {'minutes': 15,
  'rmse': 0.6296533942222595,
  'mae': 0.33992499113082886,
  'hotspot_top3_hit_rate': 0.5094055533409119,
  'hotspot_top5_recall': 0.22874343395233154},
 '6': {'minutes': 30,
  'rmse': 0.6457056403160095,
  'mae': 0.35896235704421997,
  'hotspot_top3_hit_rate': 0.5011286735534668,
  'hotspot_top5_recall': 0.23190370202064514},
 '12': {'minutes': 60,
  'rmse': 0.6226330399513245,
  'mae': 0.33097970485687256,
  'hotspot_top3_hit_rate': 0.38525205850601196,
  'hotspot_top5_recall': 0.19533486664295197}}

## ❌ REJECTED: 24-hour rolling average demand

**Verdict (2024-01 data, full-train quick-train, hidden=32/window=24/8 epochs — see chat log for the run): do not promote.**

Computed directly from `demand.npy` — no need to re-run `preprocess.py` or touch the raw CSV. Causal (trailing) window only, same log1p + train-split z-score convention `preprocess.py` already uses for the existing demand feature.

**Result:** RMSE improved consistently at all 4 horizons (+5.6% to +10.4% better), but hotspot top-3 hit rate got *worse* at all 4 horizons — mildly at 5/15 min (35%→27%, 51%→27%), badly at 30/60 min (50%→14%, 39%→15%). Likely cause: a 24h rolling mean is a slow-moving, smoothed signal that pulls predictions toward each zone's "typical" level — good for average error, but it blurs the sharp zone-to-zone differences the model needs to correctly rank *which specific 3 zones* are hottest right now. Since fleet repositioning cares about ranking, not average error, this is a net negative for the actual use case as implemented.

**Kept here as a documented negative result** (not deleted) so this exact idea doesn't get re-tested from scratch later. A shorter rolling window (e.g. 1h instead of 24h, less smoothing) might behave differently and would be a distinct experiment, not a retry of this one.

In [5]:
def add_rolling_feature(features, demand, meta, window_bins=288):
    """Append a causal rolling-mean-of-log-demand channel. window_bins=288 -> 24h at 5-min bins."""
    log_demand = np.log1p(demand)  # [Z, T]
    rolling = np.stack([
        pd.Series(log_demand[z]).rolling(window=window_bins, min_periods=1).mean().to_numpy()
        for z in range(log_demand.shape[0])
    ])
    train_end = meta["split"]["train"][1]
    mu = rolling[:, :train_end].mean(axis=1, keepdims=True)
    sigma = rolling[:, :train_end].std(axis=1, keepdims=True)
    rolling_z = (rolling - mu) / np.maximum(sigma, 1e-6)
    return np.concatenate([features, rolling_z[:, :, None].astype(np.float32)], axis=2)


features_rolling = add_rolling_feature(features, demand, meta, window_bins=288)
print("augmented features:", features_rolling.shape, "(added 1 channel: 24h rolling mean)")

augmented features: (253, 8928, 7) (added 1 channel: 24h rolling mean)


In [6]:
rolling_metrics, rolling_history = quick_train(features_rolling, meta, epochs=8)
compare(baseline_metrics, rolling_metrics, label="rolling24h")

epoch=01 train_rmse=0.7972 val_rmse=0.8097


epoch=02 train_rmse=0.7501 val_rmse=0.7761


epoch=03 train_rmse=0.7353 val_rmse=0.7771


epoch=04 train_rmse=0.7381 val_rmse=0.7660


epoch=05 train_rmse=0.7227 val_rmse=0.7613


epoch=06 train_rmse=0.7191 val_rmse=0.7597


epoch=07 train_rmse=0.7176 val_rmse=0.7590


epoch=08 train_rmse=0.7165 val_rmse=0.7585


,baseline_rmse,rolling24h_rmse,rmse_delta_%,baseline_top3_hit,rolling24h_top3_hit
minutes,,,,,
5,0.6569,0.5887,10.39,0.348,0.272
15,0.6297,0.5881,6.59,0.509,0.266
30,0.6457,0.6046,6.37,0.501,0.135
60,0.6226,0.5874,5.65,0.385,0.146


## ✅ PROMISING: dropoffs per zone — not yet implemented for real

**Verdict (2024-01 data, full-train quick-train, hidden=32/window=24/8 epochs): worth promoting, pending a full CLI confirmation run.**

RMSE improved consistently at all 4 horizons (+8.3% to +11.1%, comparable to the rejected rolling-average's gains) — but unlike rolling average, hotspot top-3 hit rate did **not** collapse: it improved at 5 min (34.8%→44.2%) and 60 min (38.5%→44.5%), stayed essentially flat at 30 min, and only dipped slightly at 15 min (50.9%→48.4%, likely noise at this reduced capacity/epoch count). No horizon regressed badly, unlike rolling average's 30/60-min collapse.

Likely why it behaves differently from rolling average: dropoffs are a fast-moving "cabs just became available here" signal, not a smoothed one — so it doesn't blur the zone-to-zone differences the model needs for ranking.

**This needed one pass over the raw CSV** (dropoff time/zone isn't in `demand.npy`) — built by `scripts/build_dropoff.py`, cached to `real_processed_265/dropoff.npy`. Sanity-checked: 2,586,749 pickups vs. 2,586,529 dropoffs (99.99% match, as expected — almost every trip has both).

**Full implementation plan for promoting this to production is written in the docstring of `scripts/build_dropoff.py`** — summary: fold the binning loop into `preprocess.py`'s existing pass-2 loop (avoids a second CSV read), add it as feature channel 6 (log1p+zscore, same convention as `demand`), update `metadata.json`'s `feature_names`, regenerate `features_clipped.npy`, then confirm with a full 50-epoch CLI run + `hotspot_eval.py` before trusting these numbers or merging to `main`.

In [7]:
dropoff_path = os.path.join(DATA_DIR, "dropoff.npy")
dropoff = np.load(dropoff_path)
print("loaded", dropoff_path, dropoff.shape,
      "| total pickups:", int(demand.sum()), "| total dropoffs:", int(dropoff.sum()))

features_dropoff = add_rolling_feature(features, dropoff, meta, window_bins=1)
print("augmented features:", features_dropoff.shape, "(added 1 channel: dropoff count, no smoothing)")

loaded C:\Users\shris\Desktop\Projects\FleetMg\SurgeMap\real_processed_265\dropoff.npy (253, 8928) | total pickups: 2586749 | total dropoffs: 2586529
augmented features: (253, 8928, 7) (added 1 channel: dropoff count, no smoothing)


In [8]:
dropoff_metrics, dropoff_history = quick_train(features_dropoff, meta, epochs=8)
compare(baseline_metrics, dropoff_metrics, label="dropoff")

epoch=01 train_rmse=0.7892 val_rmse=0.7958


epoch=02 train_rmse=0.7385 val_rmse=0.7734


epoch=03 train_rmse=0.7336 val_rmse=0.7741


epoch=04 train_rmse=0.7277 val_rmse=0.7612


epoch=05 train_rmse=0.7176 val_rmse=0.7590


epoch=06 train_rmse=0.7174 val_rmse=0.7579


epoch=07 train_rmse=0.7188 val_rmse=0.7625


epoch=08 train_rmse=0.7173 val_rmse=0.7575


,baseline_rmse,dropoff_rmse,rmse_delta_%,baseline_top3_hit,dropoff_top3_hit
minutes,,,,,
5,0.6569,0.5843,11.05,0.348,0.442
15,0.6297,0.5772,8.32,0.509,0.484
30,0.6457,0.5810,10.02,0.501,0.503
60,0.6226,0.5635,9.50,0.385,0.445


## ❌ REJECTED: weather (flat broadcast AND zone-interaction variants)

**Verdict: do not promote, either form.**

Flat broadcast (same weather value to every zone): RMSE improved consistently (+4.4% to +9.9%), but hotspot top-3 hit rate **collapsed at every horizon** (34.8%→10.1%, 50.9%→15.1%, 50.1%→13.7%, 38.5%→14.7%) — worse than even the rejected rolling-average.

Tried a fix — `precip_z[t] * zone_level_z[zone]` (see next cells), an explicit per-zone interaction so the value varies by zone, not just time — but it **also collapsed ranking** just as badly (12.9%/13.9%/12.9%/16.6%), despite still improving RMSE (+3.0% to +8.2%). In hindsight: `zone_level_z` is a fixed-per-zone constant, so the interaction still reduces to "the same time-varying global signal, rescaled by a static per-zone number" — largely redundant with what the model can already infer from each zone's own recent demand history, so it didn't supply genuinely new zone-*and*-time-varying information.

**Conclusion: weather-as-implemented isn't a good input for this model's ranking task**, even though it clearly *does* explain some genuine city-wide demand variance (the consistent RMSE gains across every variant tested confirm that part is real). Closing this line of experimentation — a real fix would need per-zone weather *sensitivity* learned from many historical storm events, which a single month of data can't support.

Source/build details kept below and in `scripts/build_weather.py`'s docstring for reference, in case revisited later with more data.

In [9]:
def add_broadcast_feature(features, series, meta):
    """Append a [T, C] global signal (already zscore'd, e.g. weather) to [Z, T, F],
    broadcasting identically to every zone -- same pattern preprocess.py uses for
    the calendar features (features[:, :, 1:] = calendar[None, :, :])."""
    Z = features.shape[0]
    tiled = np.broadcast_to(series[None, :, :], (Z, series.shape[0], series.shape[1]))
    return np.concatenate([features, tiled.astype(np.float32)], axis=2)


weather = np.load(os.path.join(DATA_DIR, "weather.npy"))  # [T, 2]: precip_log_zscore, temp_zscore
features_weather = add_broadcast_feature(features, weather, meta)
print("weather:", weather.shape, "| augmented features:", features_weather.shape,
      "(added 2 channels: precip, temp)")

weather: (8928, 2) | augmented features: (253, 8928, 8) (added 2 channels: precip, temp)


In [10]:
weather_metrics, weather_history = quick_train(features_weather, meta, epochs=8)
compare(baseline_metrics, weather_metrics, label="weather")

epoch=01 train_rmse=0.8041 val_rmse=0.8150


epoch=02 train_rmse=0.7610 val_rmse=0.7984


epoch=03 train_rmse=0.7418 val_rmse=0.7810


epoch=04 train_rmse=0.7301 val_rmse=0.7829


epoch=05 train_rmse=0.7352 val_rmse=0.8062


epoch=06 train_rmse=0.7281 val_rmse=0.7696


epoch=07 train_rmse=0.7225 val_rmse=0.7692


epoch=08 train_rmse=0.7258 val_rmse=0.7650


,baseline_rmse,weather_rmse,rmse_delta_%,baseline_top3_hit,weather_top3_hit
minutes,,,,,
5,0.6569,0.5916,9.94,0.348,0.101
15,0.6297,0.5887,6.51,0.509,0.151
30,0.6457,0.5886,8.84,0.501,0.137
60,0.6226,0.5953,4.39,0.385,0.147


## Experiment: weather × zone-level interaction

The flat broadcast above got the same weather value into every zone identically. Since the model's weights are shared across zones, a flat signal can only shift every zone's prediction by roughly the same amount (helps RMSE, doesn't help ranking, as we saw). This tests a genuinely different feature: `precip_z[t] * zone_level_z[zone]` — a product of the (uniform) weather reading and each zone's own (train-only) typical demand level, so the resulting value differs by zone even though the raw weather doesn't. Hypothesis: does rain affect busy zones and quiet zones differently? Same for temperature.

Replaces the flat weather channels (isolates the interaction's effect cleanly) rather than adding alongside them.

In [ ]:
def add_interaction_feature(features, per_zone_scale, per_time_series, meta):
    """Append channel(s) = per_zone_scale[zone] * per_time_series[t, c] -- a product
    of a zone-level value and a city-wide time series, so the result varies by BOTH
    zone and time even though per_time_series alone only varies by time."""
    tiled_time = np.broadcast_to(per_time_series[None, :, :],
                                  (features.shape[0], *per_time_series.shape))  # [Z, T, C]
    interaction = per_zone_scale[:, None, None] * tiled_time  # [Z, T, C]
    return np.concatenate([features, interaction.astype(np.float32)], axis=2)


train_end = meta["split"]["train"][1]
log_demand = np.log1p(demand)  # [Z, T]
zone_level = log_demand[:, :train_end].mean(axis=1)  # [Z] -- train-only typical demand per zone
zone_level_z = (zone_level - zone_level.mean()) / max(float(zone_level.std()), 1e-6)

features_weather_interact = add_interaction_feature(features, zone_level_z, weather, meta)
print("zone_level_z range:", zone_level_z.min(), "to", zone_level_z.max())
print("augmented features:", features_weather_interact.shape,
      "(added 2 channels: precip x zone_level, temp x zone_level)")

In [ ]:
weather_interact_metrics, weather_interact_history = quick_train(features_weather_interact, meta, epochs=8)
compare(baseline_metrics, weather_interact_metrics, label="weather_x_zone")

## Promoting a feature for real

If a candidate shows a **consistent** improvement here (lower RMSE *and* higher top-3 hit rate across most horizons, not just one lucky horizon):

1. Wire the feature into `scripts/preprocess.py` for real (so it's part of the reproducible pipeline, not just this notebook).
2. Re-run `preprocess.py` to regenerate `features_clipped.npy` with the new channel baked in.
3. Run the full CLI training: `python scripts/train_multihorizon_torch.py --data-dir real_processed_265 ...` (50 epochs, real early stopping).
4. Confirm with `python scripts/hotspot_eval.py ...` — that's the number that actually counts, not the quick-train number above.
5. Only then consider merging `feature-experiments` back into `main`.